In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path(r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump")

INPUT_FILE = (
    PROJECT_ROOT / "data" / "processed" / "baseline" /
    "crop_soil_climate_with_10yr_baseline_2013_2025.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT / "data" / "processed" / "financial" /
    "crop_soil_climate_with_revenue_success_2013_2025.csv"
)

AUDIT_FILE = (
    PROJECT_ROOT / "data" / "processed" / "financial" /
    "msp_revenue_success_audit.csv"
)

print("Input :", INPUT_FILE)
print("Output:", OUTPUT_FILE)
print("Audit :", AUDIT_FILE)

Input : C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\baseline\crop_soil_climate_with_10yr_baseline_2013_2025.csv
Output: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\financial\crop_soil_climate_with_revenue_success_2013_2025.csv
Audit : C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\financial\msp_revenue_success_audit.csv


In [2]:
df = pd.read_csv(INPUT_FILE)

print("Shape:", df.shape)
print("Columns:", len(df.columns))

required = [
    "year", "state", "district", "crop", "yield_kg_ha",
    "historical_10yr_baseline_yield_kg_ha"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["year"] = pd.to_numeric(df["year"], errors="raise").astype(int)
df["yield_kg_ha"] = pd.to_numeric(df["yield_kg_ha"], errors="coerce")
df["historical_10yr_baseline_yield_kg_ha"] = pd.to_numeric(
    df["historical_10yr_baseline_yield_kg_ha"], errors="coerce"
)

df["crop"] = df["crop"].astype("string").str.strip()

EXPECTED_YEARS = set(range(2013, 2025))
EXPECTED_CROPS = {
    "Arhar/Tur", "Bajra", "Gram", "Groundnut", "Jowar", "Maize",
    "Ragi", "Rice", "Soyabean", "Sugarcane", "Urad", "Wheat"
}

assert set(df["year"].unique()) == EXPECTED_YEARS
assert set(df["crop"].dropna().unique()) == EXPECTED_CROPS
assert len(df) == 67826
assert (df["yield_kg_ha"].dropna() >= 0).all()
assert (df["historical_10yr_baseline_yield_kg_ha"].dropna() >= 0).all()

print("PASS: input validation")
print("Years:", sorted(df["year"].unique()))
print("Crops:", sorted(df["crop"].unique()))
print("Missing actual yield:", int(df["yield_kg_ha"].isna().sum()))
print("Missing historical baseline:", int(df["historical_10yr_baseline_yield_kg_ha"].isna().sum()))


Shape: (67826, 58)
Columns: 58
PASS: input validation
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Crops: ['Arhar/Tur', 'Bajra', 'Gram', 'Groundnut', 'Jowar', 'Maize', 'Ragi', 'Rice', 'Soyabean', 'Sugarcane', 'Urad', 'Wheat']
Missing actual yield: 2355
Missing historical baseline: 8169


In [3]:
#yield unit conversion
HA_TO_ACRE = 2.47105381
KG_TO_QUINTAL = 1 / 100

df["yield_q_acre"] = (
    df["yield_kg_ha"] * HA_TO_ACRE * KG_TO_QUINTAL
)

df["historical_baseline_yield_q_acre"] = (
    df["historical_10yr_baseline_yield_kg_ha"] *
    HA_TO_ACRE * KG_TO_QUINTAL
)

assert (df["yield_q_acre"].dropna() >= 0).all()
assert (df["historical_baseline_yield_q_acre"].dropna() >= 0).all()

display(
    df[
        ["crop", "year", "yield_kg_ha", "yield_q_acre",
         "historical_10yr_baseline_yield_kg_ha",
         "historical_baseline_yield_q_acre"]
    ].head(10)
)

print("PASS: yield converted to Q/acre")


,crop,year,yield_kg_ha,yield_q_acre,historical_10yr_baseline_yield_kg_ha,historical_baseline_yield_q_acre
0,Maize,2013,526.0,12.997743,NaN,NaN
1,Maize,2014,9167.0,226.521503,526.000000,12.997743
2,Maize,2015,1000.0,24.710538,4846.500000,119.759623
3,Maize,2021,1133.0,27.997040,3564.333333,88.076595
4,Maize,2023,NaN,NaN,2956.500000,73.056706
5,Maize,2024,NaN,NaN,3766.666667,93.076360
6,Rice,2013,2377.0,58.736949,NaN,NaN
7,Rice,2014,2348.0,58.020343,2377.000000,58.736949
8,Rice,2015,2240.0,55.351605,2362.500000,58.378646
9,Rice,2016,1430.0,35.336069,2321.666667,57.369633


PASS: yield converted to Q/acre


In [4]:
#Historical MSP / FRP table

# Government of India support prices, ₹/quintal.
# Year represents the agricultural/marketing season starting year.
# 2013-2020 values are based on Government MSP tables/PIB releases.
# 2021-2024 values are from Government MSP tables/PIB releases.

PRICE_DATA = {
    "Arhar/Tur": {
        2013: 4300, 2014: 4350, 2015: 4625, 2016: 5050,
        2017: 5450, 2018: 5675, 2019: 5800, 2020: 6000,
        2021: 6300, 2022: 6600, 2023: 7000, 2024: 7550
    },
    "Bajra": {
        2013: 1250, 2014: 1250, 2015: 1275, 2016: 1330,
        2017: 1425, 2018: 1950, 2019: 2000, 2020: 2150,
        2021: 2250, 2022: 2350, 2023: 2500, 2024: 2625
    },
    "Gram": {
        2013: 3100, 2014: 3175, 2015: 3500, 2016: 4000,
        2017: 4400, 2018: 4620, 2019: 4875, 2020: 5100,
        2021: 5100, 2022: 5230, 2023: 5335, 2024: 5440
    },
    "Groundnut": {
        2013: 4000, 2014: 4000, 2015: 4030, 2016: 4220,
        2017: 4450, 2018: 4890, 2019: 5090, 2020: 5275,
        2021: 5550, 2022: 5850, 2023: 6377, 2024: 6783
    },
    "Jowar": {
        2013: 1500, 2014: 1530, 2015: 1570, 2016: 1625,
        2017: 1700, 2018: 2430, 2019: 2550, 2020: 2620,
        2021: 2738, 2022: 2970, 2023: 3180, 2024: 3371
    },
    "Maize": {
        2013: 1310, 2014: 1310, 2015: 1325, 2016: 1365,
        2017: 1425, 2018: 1700, 2019: 1760, 2020: 1850,
        2021: 1870, 2022: 1962, 2023: 2090, 2024: 2225
    },
    "Ragi": {
        2013: 1500, 2014: 1550, 2015: 1650, 2016: 1725,
        2017: 1900, 2018: 2897, 2019: 3150, 2020: 3295,
        2021: 3377, 2022: 3578, 2023: 3846, 2024: 4290
    },
    "Rice": {
        2013: 1310, 2014: 1360, 2015: 1410, 2016: 1470,
        2017: 1550, 2018: 1750, 2019: 1815, 2020: 1868,
        2021: 1940, 2022: 2040, 2023: 2183, 2024: 2300
    },
    "Soyabean": {
        2013: 2560, 2014: 2560, 2015: 2600, 2016: 2775,
        2017: 3050, 2018: 3399, 2019: 3710, 2020: 3880,
        2021: 3950, 2022: 4300, 2023: 4600, 2024: 4892
    },
    "Urad": {
        2013: 4300, 2014: 4350, 2015: 4625, 2016: 5000,
        2017: 5400, 2018: 5600, 2019: 5700, 2020: 6000,
        2021: 6300, 2022: 6600, 2023: 6950, 2024: 7400
    },
    "Wheat": {
        2013: 1400, 2014: 1450, 2015: 1525, 2016: 1625,
        2017: 1735, 2018: 1840, 2019: 1925, 2020: 1975,
        2021: 2015, 2022: 2015, 2023: 2125, 2024: 2275
    },
    "Sugarcane": {
        2013: 210, 2014: 220, 2015: 230, 2016: 230,
        2017: 255, 2018: 275, 2019: 275, 2020: 285,
        2021: 290, 2022: 305, 2023: 315, 2024: 340
    }
}

PRICE_TYPE = {crop: "MSP" for crop in PRICE_DATA}
PRICE_TYPE["Sugarcane"] = "FRP"

price_rows = []
for crop, year_values in PRICE_DATA.items():
    for year, price in year_values.items():
        price_rows.append({
            "crop": crop,
            "year": year,
            "support_price_rs_per_quintal": price,
            "support_price_type": PRICE_TYPE[crop]
        })

price_df = pd.DataFrame(price_rows)

assert len(price_df) == 12 * 12
assert price_df.duplicated(["crop", "year"]).sum() == 0
assert set(price_df["crop"]) == EXPECTED_CROPS
assert set(price_df["year"]) == EXPECTED_YEARS
assert (price_df["support_price_rs_per_quintal"] > 0).all()

display(price_df.head(20))
print("PASS: support-price table contains all 12 crops × 12 years")


,crop,year,support_price_rs_per_quintal,support_price_type
0,Arhar/Tur,2013,4300,MSP
1,Arhar/Tur,2014,4350,MSP
2,Arhar/Tur,2015,4625,MSP
3,Arhar/Tur,2016,5050,MSP
4,Arhar/Tur,2017,5450,MSP
5,Arhar/Tur,2018,5675,MSP
6,Arhar/Tur,2019,5800,MSP
7,Arhar/Tur,2020,6000,MSP
8,Arhar/Tur,2021,6300,MSP
9,Arhar/Tur,2022,6600,MSP


PASS: support-price table contains all 12 crops × 12 years


In [5]:
df = df.merge(
    price_df,
    on=["crop", "year"],
    how="left",
    validate="many_to_one"
)

assert df["support_price_rs_per_quintal"].notna().all()
assert df["support_price_type"].notna().all()

print("Support-price coverage:", df["support_price_rs_per_quintal"].notna().mean())
print(df["support_price_type"].value_counts())


Support-price coverage: 1.0
support_price_type
MSP    61671
FRP     6155
Name: count, dtype: int64


In [6]:
#Revenue calculation
df["actual_revenue_rs_per_acre"] = (
    df["yield_q_acre"] *
    df["support_price_rs_per_quintal"]
)

df["target_revenue_rs_per_acre"] = (
    df["historical_baseline_yield_q_acre"] *
    df["support_price_rs_per_quintal"]
)

df["revenue_vs_target_ratio"] = np.where(
    df["target_revenue_rs_per_acre"] > 0,
    df["actual_revenue_rs_per_acre"] /
    df["target_revenue_rs_per_acre"],
    np.nan
)

# Success label:
# 1 = actual revenue meets/exceeds historical target
# 0 = actual revenue is below historical target
# NaN = label cannot be safely determined because actual yield or target baseline is missing
df["success_label"] = np.where(
    df["actual_revenue_rs_per_acre"].notna() &
    df["target_revenue_rs_per_acre"].notna() &
    (df["target_revenue_rs_per_acre"] > 0),
    (df["actual_revenue_rs_per_acre"] >= df["target_revenue_rs_per_acre"]).astype(int),
    np.nan
)

df["financial_label_status"] = np.select(
    [
        df["yield_q_acre"].isna(),
        df["historical_baseline_yield_q_acre"].isna(),
        df["target_revenue_rs_per_acre"].fillna(0).eq(0),
    ],
    [
        "missing_actual_yield",
        "missing_historical_baseline",
        "invalid_target",
    ],
    default="label_available"
)

display(
    df[
        [
            "year", "crop", "yield_kg_ha", "yield_q_acre",
            "historical_baseline_yield_q_acre",
            "support_price_rs_per_quintal", "support_price_type",
            "actual_revenue_rs_per_acre",
            "target_revenue_rs_per_acre",
            "revenue_vs_target_ratio",
            "success_label",
            "financial_label_status"
        ]
    ].head(15)
)


,year,crop,yield_kg_ha,yield_q_acre,historical_baseline_yield_q_acre,support_price_rs_per_quintal,support_price_type,actual_revenue_rs_per_acre,target_revenue_rs_per_acre,revenue_vs_target_ratio,success_label,financial_label_status
0,2013,Maize,526.0,12.997743,NaN,1310,MSP,17027.043383,NaN,NaN,NaN,missing_historical_baseline
1,2014,Maize,9167.0,226.521503,12.997743,1310,MSP,296743.168619,17027.043383,17.427757,1.0,label_available
2,2015,Maize,1000.0,24.710538,119.759623,1325,MSP,32741.462983,158681.500345,0.206334,0.0,label_available
3,2021,Maize,1133.0,27.997040,88.076595,1870,MSP,52354.464178,164703.231966,0.317872,0.0,label_available
4,2023,Maize,NaN,NaN,73.056706,2090,MSP,NaN,152688.515316,NaN,NaN,missing_actual_yield
5,2024,Maize,NaN,NaN,93.076360,2225,MSP,NaN,207094.901393,NaN,NaN,missing_actual_yield
6,2013,Rice,2377.0,58.736949,NaN,1310,MSP,76945.403273,NaN,NaN,NaN,missing_historical_baseline
7,2014,Rice,2348.0,58.020343,58.736949,1360,MSP,78907.667104,79882.250727,0.987800,0.0,label_available
8,2015,Rice,2240.0,55.351605,58.378646,1410,MSP,78045.763535,82313.891228,0.948148,0.0,label_available
9,2016,Rice,1430.0,35.336069,57.369633,1470,MSP,51944.022140,84333.359955,0.615937,0.0,label_available


In [7]:
print("Financial label status:")
print(df["financial_label_status"].value_counts())

print("\nSuccess label distribution:")
print(df["success_label"].value_counts(dropna=False))

print("\nSuccess rate among available labels:")
available = df["success_label"].notna()
if available.any():
    print(df.loc[available, "success_label"].mean())

print("\nRevenue ratio:")
print(df["revenue_vs_target_ratio"].describe())


Financial label status:
financial_label_status
label_available                58185
missing_historical_baseline     7217
missing_actual_yield            2355
invalid_target                    69
Name: count, dtype: int64

Success label distribution:
success_label
1.0    35878
0.0    22307
NaN     9641
Name: count, dtype: int64

Success rate among available labels:
0.6166194036263641

Revenue ratio:
count    58185.000000
mean         1.140106
std          5.348166
min          0.000000
25%          0.920090
50%          1.042880
75%          1.210322
max       1200.643599
Name: revenue_vs_target_ratio, dtype: float64


In [8]:
#Financial sanity checks
assert len(df) == 67826
assert (df["yield_q_acre"].dropna() >= 0).all()
assert (df["historical_baseline_yield_q_acre"].dropna() >= 0).all()
assert (df["support_price_rs_per_quintal"] > 0).all()
assert (df["actual_revenue_rs_per_acre"].dropna() >= 0).all()
assert (df["target_revenue_rs_per_acre"].dropna() >= 0).all()

valid_labels = df["success_label"].dropna()
assert set(valid_labels.unique()).issubset({0.0, 1.0})

assert (df.loc[
    df["success_label"].notna(),
    ["actual_revenue_rs_per_acre", "target_revenue_rs_per_acre"]
].notna().all(axis=1)).all()

assert (df.loc[
    df["crop"] == "Sugarcane",
    "support_price_type"
] == "FRP").all()

assert (df.loc[
    df["crop"] != "Sugarcane",
    "support_price_type"
] == "MSP").all()

print("PASS: financial sanity checks")


PASS: financial sanity checks


In [9]:
# Financial audit
audit = pd.DataFrame({
    "metric": [
        "input_output_rows",
        "price_table_rows",
        "crops",
        "years",
        "actual_yield_missing",
        "historical_baseline_missing",
        "actual_revenue_available",
        "target_revenue_available",
        "success_label_available",
        "success_label_1",
        "success_label_0",
        "msp_rows",
        "frp_rows"
    ],
    "value": [
        len(df),
        len(price_df),
        df["crop"].nunique(),
        df["year"].nunique(),
        int(df["yield_q_acre"].isna().sum()),
        int(df["historical_baseline_yield_q_acre"].isna().sum()),
        int(df["actual_revenue_rs_per_acre"].notna().sum()),
        int(df["target_revenue_rs_per_acre"].notna().sum()),
        int(df["success_label"].notna().sum()),
        int((df["success_label"] == 1).sum()),
        int((df["success_label"] == 0).sum()),
        int((df["support_price_type"] == "MSP").sum()),
        int((df["support_price_type"] == "FRP").sum())
    ]
})

display(audit)


,metric,value
0,input_output_rows,67826
1,price_table_rows,144
2,crops,12
3,years,12
4,actual_yield_missing,2355
5,historical_baseline_missing,8169
6,actual_revenue_available,65471
7,target_revenue_available,59657
8,success_label_available,58185
9,success_label_1,35878


In [10]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
AUDIT_FILE.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_FILE, index=False)
audit.to_csv(AUDIT_FILE, index=False)

print("Saved dataset:", OUTPUT_FILE)
print("Saved audit  :", AUDIT_FILE)
print("Final shape  :", df.shape)


Saved dataset: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\financial\crop_soil_climate_with_revenue_success_2013_2025.csv
Saved audit  : C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\financial\msp_revenue_success_audit.csv
Final shape  : (67826, 67)
